# Lung Cancer Classification — EfficientNetB0 (Clean Version)

This notebook keeps the original dataset/split approach but removes the broken and duplicated augmentation/model cells.

**Pipeline:** dataset → stratified 70/15/15 split → training-only balancing/augmentation → EfficientNetB0 transfer learning → fine-tuning → test evaluation → ROC-AUC → error analysis.


In [ ]:
# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from tensorflow.keras.applications import EfficientNetB0

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU(s):", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# STEP 2: DATASET PATH AND CLASS DEFINITIONS
# ============================================================

ROOT = "/kaggle/input/datasets/adityamahimkar/iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset"

# NOTE: "Bengin cases" is intentionally spelled this way because
# that is the folder spelling in the dataset.
CLASS_DIRS = {
    "Normal": "Normal cases",
    "Benign": "Bengin cases",
    "Malignant": "Malignant cases"
}

CLASS_NAMES = ["Normal", "Benign", "Malignant"]
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

print("Dataset root:", ROOT)
print("Classes:", CLASS_NAMES)

for label, folder in CLASS_DIRS.items():
    folder_path = os.path.join(ROOT, folder)
    print(f"{label:10s}: {folder_path} | exists = {os.path.isdir(folder_path)}")
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Missing class folder: {folder_path}")

In [ ]:
# ============================================================
# STEP 3: CREATE CLEAN DATAFRAME
# ============================================================

records = []

for label in CLASS_NAMES:
    folder_path = os.path.join(ROOT, CLASS_DIRS[label])

    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(VALID_EXTENSIONS):
            records.append({
                "filepath": os.path.join(folder_path, filename),
                "label": label
            })

df = pd.DataFrame(records)

# Remove duplicate paths
df = df.drop_duplicates(subset="filepath").reset_index(drop=True)

if df.empty:
    raise ValueError("No images were found. Check ROOT and CLASS_DIRS.")

print("Total original images:", len(df))
print("\nOriginal class distribution:")
print(df["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))

display(df.head())

In [ ]:
# ============================================================
# STEP 4: ORIGINAL CLASS DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="label", order=CLASS_NAMES)
plt.title("Original Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5: STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# 70% TRAIN | 15% VALIDATION | 15% TEST
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    counts = split_df["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
    print(f"\n{name}: {len(split_df)} images")
    print(counts)

In [ ]:
# ============================================================
# STEP 6: CHECK FOR FILEPATH LEAKAGE
# ============================================================

train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

print("Train ∩ Validation:", len(train_paths & val_paths))
print("Train ∩ Test:", len(train_paths & test_paths))
print("Validation ∩ Test:", len(val_paths & test_paths))

assert len(train_paths & val_paths) == 0
assert len(train_paths & test_paths) == 0
assert len(val_paths & test_paths) == 0

print("\nSUCCESS: No filepath overlap between splits.")

In [ ]:
# ============================================================
# STEP 7: CREATE BALANCED TRAINING DATASET
# ============================================================
# IMPORTANT:
# Only TRAIN images are copied/augmented.
# Validation and TEST images are never augmented or copied here.

IMG_SIZE = (224, 224)
BALANCED_ROOT = "/kaggle/working/balanced_augmented_train"

if os.path.exists(BALANCED_ROOT):
    shutil.rmtree(BALANCED_ROOT)

for class_name in CLASS_NAMES:
    os.makedirs(os.path.join(BALANCED_ROOT, class_name), exist_ok=True)

train_counts = (
    train_df["label"]
    .value_counts()
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)

if (train_counts == 0).any():
    raise ValueError("At least one class has zero training images.")

# Balance to the size of the largest original training class.
TARGET_PER_CLASS = int(train_counts.max())

print("Original training counts:")
print(train_counts)
print("\nTarget images per class:", TARGET_PER_CLASS)

# Conservative augmentation for medical images.
# No vertical flip; no extreme transformations.
augmentation_generator = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08,
    shear_range=0.05,
    horizontal_flip=False,
    fill_mode="nearest"
)

for class_name in CLASS_NAMES:

    class_df = (
        train_df[train_df["label"] == class_name]
        .reset_index(drop=True)
    )

    output_class_dir = os.path.join(BALANCED_ROOT, class_name)

    # Copy original training images
    for idx, row in class_df.iterrows():

        source_path = row["filepath"]

        if not os.path.isfile(source_path):
            raise FileNotFoundError(f"Training image not found: {source_path}")

        extension = Path(source_path).suffix.lower()

        destination_path = os.path.join(
            output_class_dir,
            f"original_{idx:06d}{extension}"
        )

        shutil.copy2(source_path, destination_path)

    current_count = len(class_df)
    aug_index = 0

    # Generate only enough augmented images to balance the class
    while current_count < TARGET_PER_CLASS:

        row = class_df.iloc[aug_index % len(class_df)]
        source_path = row["filepath"]

        image = tf.keras.utils.load_img(
            source_path,
            target_size=IMG_SIZE
        )

        image_array = tf.keras.utils.img_to_array(image)
        image_array = np.expand_dims(image_array, axis=0)

        batch = next(
            augmentation_generator.flow(
                image_array,
                batch_size=1,
                shuffle=False,
                seed=SEED + aug_index
            )
        )

        augmented_image = tf.keras.utils.array_to_img(batch[0])

        output_path = os.path.join(
            output_class_dir,
            f"augmented_{aug_index:06d}.jpg"
        )

        augmented_image.save(output_path, quality=95)

        current_count += 1
        aug_index += 1

    print(f"{class_name:10s}: {current_count} images created")

print("\nSUCCESS: Balanced training dataset created.")

In [ ]:
# ============================================================
# STEP 8: BUILD DATAFRAME FOR BALANCED TRAINING SET
# ============================================================

balanced_records = []

for class_name in CLASS_NAMES:

    class_dir = os.path.join(BALANCED_ROOT, class_name)

    for filename in sorted(os.listdir(class_dir)):

        if filename.lower().endswith(VALID_EXTENSIONS):

            balanced_records.append({
                "filepath": os.path.join(class_dir, filename),
                "label": class_name
            })

balanced_train_df = pd.DataFrame(balanced_records)

balanced_counts = (
    balanced_train_df["label"]
    .value_counts()
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)

print("Balanced training dataset size:", len(balanced_train_df))
print("\nBalanced training class distribution:")
print(balanced_counts)

assert balanced_counts.min() > 0
assert balanced_counts.nunique() == 1

print("\nSUCCESS: All classes are exactly balanced.")

In [ ]:
# ============================================================
# STEP 9: BEFORE VS AFTER BALANCING
# ============================================================

comparison_df = pd.DataFrame({
    "Original Training": train_df["label"].value_counts().reindex(CLASS_NAMES),
    "Balanced Training": balanced_train_df["label"].value_counts().reindex(CLASS_NAMES)
}).fillna(0).astype(int)

display(comparison_df)

comparison_df.plot(kind="bar", figsize=(10, 6))
plt.title("Training Class Distribution Before and After Balancing")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 10: IMAGE PARAMETERS
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(CLASS_NAMES)

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)

In [ ]:
# ============================================================
# STEP 11: DATA GENERATORS
# ============================================================
# The saved training set has already been balanced/augmented.
# Therefore we do NOT augment it again here.
#
# Validation and test are never augmented.
#
# Keras EfficientNetB0 includes its expected input rescaling,
# so rescale=1/255 is intentionally not added here.

train_datagen = ImageDataGenerator()
val_test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_dataframe(
    dataframe=balanced_train_df,
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=True,
    seed=SEED
)

validation_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False
)

print("\nClass indices:")
print(train_generator.class_indices)

assert train_generator.class_indices == {
    "Normal": 0,
    "Benign": 1,
    "Malignant": 2
}

In [ ]:
# ============================================================
# STEP 12: VISUALIZE TRAINING IMAGES
# ============================================================

images, labels = next(train_generator)

plt.figure(figsize=(12, 10))

for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].astype("uint8"))
    label_index = np.argmax(labels[i])
    plt.title(CLASS_NAMES[label_index])
    plt.axis("off")

plt.suptitle("Samples from Balanced Training Dataset")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 13: BUILD EFFICIENTNETB0
# ============================================================

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

# Stage 1: freeze pretrained feature extractor
base_model.trainable = False

inputs = layers.Input(
    shape=(*IMG_SIZE, 3),
    name="input_image"
)

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.BatchNormalization(name="classifier_bn")(x)
x = layers.Dropout(0.35, name="dropout_1")(x)
x = layers.Dense(
    256,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    name="dense_256"
)(x)
x = layers.Dropout(0.30, name="dropout_2")(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classifier"
)(x)

model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="EfficientNetB0_LungCancer"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy")
    ]
)

model.summary()

In [ ]:
# ============================================================
# STEP 14: CALLBACKS
# ============================================================

MODEL_PATH = "/kaggle/working/best_efficientnetb0.keras"

callbacks = [
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.25,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks ready.")

In [ ]:
# ============================================================
# STEP 15: TRAIN CLASSIFICATION HEAD
# ============================================================

history_head = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# STEP 16: TRAINING CURVES
# ============================================================

def plot_history(history, title_prefix):
    history_dict = history.history

    plt.figure(figsize=(8, 5))
    plt.plot(history_dict["accuracy"], label="Training Accuracy")
    plt.plot(history_dict["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix}: Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_dict["loss"], label="Training Loss")
    plt.plot(history_dict["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix}: Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(history_head, "EfficientNetB0 Head Training")

In [ ]:
# ============================================================
# STEP 17: FINE-TUNE LAST PART OF EFFICIENTNETB0
# ============================================================

# Load the best checkpoint from Stage 1
model = tf.keras.models.load_model(MODEL_PATH)

# Find the nested EfficientNet model by its name
base_model = model.get_layer("efficientnetb0")

base_model.trainable = True

# Freeze all but the last 40 layers.
fine_tune_from = max(0, len(base_model.layers) - 40)

for i, layer in enumerate(base_model.layers):

    if i < fine_tune_from:
        layer.trainable = False

    elif isinstance(layer, layers.BatchNormalization):
        # Keep BatchNorm frozen during fine-tuning for stability.
        layer.trainable = False

    else:
        layer.trainable = True

trainable_count = sum(
    int(layer.trainable)
    for layer in base_model.layers
)

print("Total EfficientNet layers:", len(base_model.layers))
print("Trainable EfficientNet layers:", trainable_count)

In [ ]:
# ============================================================
# STEP 18: RECOMPILE FOR FINE-TUNING
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy")
    ]
)

model.summary()

In [ ]:
# ============================================================
# STEP 19: FINE-TUNE
# ============================================================

fine_tune_callbacks = [
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.25,
        patience=3,
        min_lr=1e-8,
        verbose=1
    )
]

history_finetune = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=25,
    callbacks=fine_tune_callbacks,
    verbose=1
)

plot_history(history_finetune, "EfficientNetB0 Fine-Tuning")

In [ ]:
# ============================================================
# STEP 20: LOAD BEST MODEL
# ============================================================

best_model = tf.keras.models.load_model(MODEL_PATH)

print("Best EfficientNetB0 model loaded successfully.")

In [ ]:
# ============================================================
# STEP 21: FINAL TEST EVALUATION
# ============================================================

test_generator.reset()

test_results = best_model.evaluate(
    test_generator,
    verbose=1,
    return_dict=True
)

print("\n======================================")
print("FINAL EFFICIENTNETB0 TEST RESULTS")
print("======================================")

for metric_name, value in test_results.items():
    print(f"{metric_name:15s}: {value:.4f}")

print(f"Test Accuracy (%): {test_results['accuracy'] * 100:.2f}")

In [ ]:
# ============================================================
# STEP 22: TEST PREDICTIONS
# ============================================================

test_generator.reset()

y_prob = best_model.predict(
    test_generator,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes

print("True labels:", len(y_true))
print("Predictions:", len(y_pred))

assert len(y_true) == len(y_pred)
assert y_prob.shape[0] == len(y_true)

In [ ]:
# ============================================================
# STEP 23: CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# STEP 24: CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("EfficientNetB0 Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 25: FINAL METRICS
# ============================================================

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

display(results)

In [ ]:
# ============================================================
# STEP 26: MULTICLASS ROC-AUC
# ============================================================

y_true_onehot = tf.keras.utils.to_categorical(
    y_true,
    num_classes=NUM_CLASSES
)

macro_auc = roc_auc_score(
    y_true_onehot,
    y_prob,
    multi_class="ovr",
    average="macro"
)

per_class_auc = roc_auc_score(
    y_true_onehot,
    y_prob,
    multi_class="ovr",
    average=None
)

print("Macro ROC-AUC:", round(macro_auc, 4))

auc_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "ROC-AUC": per_class_auc
})

display(auc_table)

In [ ]:
# ============================================================
# STEP 27: ERROR ANALYSIS
# ============================================================

error_df = test_df.copy().reset_index(drop=True)

error_df["True"] = [
    CLASS_NAMES[i] for i in y_true
]

error_df["Predicted"] = [
    CLASS_NAMES[i] for i in y_pred
]

error_df["Confidence"] = np.max(y_prob, axis=1)

errors = (
    error_df[error_df["True"] != error_df["Predicted"]]
    .sort_values("Confidence", ascending=False)
    .reset_index(drop=True)
)

print("Number of incorrect predictions:", len(errors))

display(errors.head(20))

In [ ]:
# ============================================================
# STEP 28: SAVE FINAL MODEL AND METRICS
# ============================================================

FINAL_MODEL_PATH = "/kaggle/working/final_lung_cancer_efficientnetb0.keras"
METRICS_PATH = "/kaggle/working/final_metrics.csv"

best_model.save(FINAL_MODEL_PATH)
results.to_csv(METRICS_PATH, index=False)

print("Final model:", FINAL_MODEL_PATH)
print("Metrics CSV:", METRICS_PATH)